# 11 - Separabilitas ruang 9 fitur SFM (Paper 4, revisi reviewer poin 1)

**Tujuan:** memberi **bukti empiris visual** untuk klaim di naskah bahwa 9 fitur
SFM kurang memisahkan NORMAL vs SERANGAN (menjelaskan kenapa one-class AE gagal,
nb10). Reviewer meminta plot t-SNE/PCA + bukti overlap tinggi.

**Output (untuk gambar di naskah + S3 `results/separability/`):**
- `sep_pca_cic.png`, `sep_pca_unsw.png` : proyeksi PCA 2D normal vs attack per dataset.
- `sep_tsne_cic.png`, `sep_tsne_unsw.png` : proyeksi t-SNE 2D (subsample).
- `sep_overlap_hist.png` : histogram skor 1D (proyeksi LDA) normal vs attack.
- `separability_metrics.csv` : AUC per-fitur + silhouette + overlap (angka untuk teks).

**Metodologi (jujur):** t-SNE pada subsample (≤4000/kelas) agar wajar waktunya;
PCA & metrik pada sampel lebih besar. Semua ANGKA NYATA dari data; jangan mengarang.

> Jalankan setelah nb01 (butuh `fed_out/{cic,unsw}_train.npz`).

## 0. Setup

In [ ]:
import importlib, sys, subprocess
for m in ('numpy','pandas','scikit-learn','matplotlib'):
    if importlib.util.find_spec(m.replace('scikit-learn','sklearn')) is None:
        subprocess.run([sys.executable,'-m','pip','install','-q',m],check=True)
import os, numpy as np, pandas as pd
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, silhouette_score
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
OUTDIR='fed_out'; os.makedirs(OUTDIR, exist_ok=True)
SEED=42; FEATS=['duration','fwd_pkts','bwd_pkts','fwd_bytes','bwd_bytes','fwd_mean','bwd_mean','src_load','dst_load']
print('setup ok')

## 1. Muat partisi (dari nb01)

In [ ]:
def load(name):
    p=os.path.join(OUTDIR,name)
    if not os.path.exists(p): print('MISSING',p); return None,None
    d=np.load(p,allow_pickle=True); return np.asarray(d['X'],float), np.asarray(d['y'],int)
data={}
for ds,f in {'CIC':'cic_train.npz','UNSW':'unsw_train.npz'}.items():
    X,y=load(f)
    if X is not None: data[ds]=(X,y); print(f'{ds}: {X.shape} pos {round(float(y.mean()),3)}')
ok=len(data)>0
if not ok: print('(jalankan nb01 dulu)')

## 2. Metrik separabilitas (ANGKA untuk teks naskah)

- **AUC per-fitur:** seberapa jauh satu fitur sendirian memisahkan normal/attack
  (0.5 = tak memisahkan). - **Silhouette** label pada ruang 9-dim ter-standarisasi.
- **Overlap (OVL)** pada proyeksi LDA 1D: makin tinggi = makin bertumpuk.

In [ ]:
def overlap_1d(a, b, bins=100):
    lo=min(a.min(),b.min()); hi=max(a.max(),b.max())
    ha,_=np.histogram(a,bins=bins,range=(lo,hi),density=True)
    hb,_=np.histogram(b,bins=bins,range=(lo,hi),density=True)
    w=(hi-lo)/bins
    return float(np.sum(np.minimum(ha,hb))*w)  # 0..1, 1=identik total
rows=[]
if ok:
    for ds,(X,y) in data.items():
        Xs=StandardScaler().fit_transform(X)
        # AUC per fitur
        for j,fn in enumerate(FEATS):
            try: auc=roc_auc_score(y, Xs[:,j]); auc=max(auc,1-auc)
            except Exception: auc=float('nan')
            rows.append({'dataset':ds,'feature':fn,'auc_sep':round(auc,4)})
        # silhouette (subsample agar cepat)
        r=np.random.RandomState(SEED); idx=r.choice(len(y), min(5000,len(y)), replace=False)
        try: sil=silhouette_score(Xs[idx], y[idx])
        except Exception: sil=float('nan')
        # LDA 1D overlap
        z=LinearDiscriminantAnalysis(n_components=1).fit_transform(Xs,y).ravel()
        ovl=overlap_1d(z[y==0], z[y==1])
        rows.append({'dataset':ds,'feature':'<silhouette_9d>','auc_sep':round(float(sil),4)})
        rows.append({'dataset':ds,'feature':'<LDA1d_overlap>','auc_sep':round(ovl,4)})
        print(f'{ds}: silhouette={sil:.4f}  LDA-overlap={ovl:.4f}')
    dfm=pd.DataFrame(rows); dfm.to_csv(os.path.join(OUTDIR,'separability_metrics.csv'),index=False)
    import IPython.display as ipd; ipd.display(dfm)
print('=== SEL 2 (metrik) SELESAI ===')

## 3. PCA 2D normal vs attack (per dataset)

In [ ]:
def scatter2d(Z, y, title, path):
    plt.figure(figsize=(4.2,3.6))
    for lab,mk,cl,nm in [(0,'o','#2c7fb8','normal'),(1,'x','#d7301f','attack')]:
        m=(y==lab); plt.scatter(Z[m,0],Z[m,1],s=4,marker=mk,c=cl,alpha=0.35,label=nm,linewidths=0.4)
    plt.title(title,fontsize=10); plt.legend(fontsize=8,markerscale=2)
    plt.xlabel('comp-1',fontsize=9); plt.ylabel('comp-2',fontsize=9); plt.tight_layout()
    plt.savefig(path,dpi=300); plt.close(); print('saved',path)
if ok:
    for ds,(X,y) in data.items():
        Xs=StandardScaler().fit_transform(X)
        r=np.random.RandomState(SEED); idx=r.choice(len(y), min(8000,len(y)), replace=False)
        Z=PCA(n_components=2,random_state=SEED).fit_transform(Xs[idx])
        scatter2d(Z, y[idx], f'PCA 9-SFM: {ds}', os.path.join(OUTDIR,f'sep_pca_{ds.lower()}.png'))
print('=== SEL 3 (PCA) SELESAI ===')

## 4. t-SNE 2D (subsample agar wajar waktunya)

In [ ]:
if ok:
    for ds,(X,y) in data.items():
        Xs=StandardScaler().fit_transform(X)
        r=np.random.RandomState(SEED)
        # subsample seimbang per kelas (<=4000/kelas) agar t-SNE cepat & tak bias ukuran
        i0=np.where(y==0)[0]; i1=np.where(y==1)[0]
        n=min(4000,len(i0),len(i1))
        sel=np.concatenate([r.choice(i0,n,replace=False), r.choice(i1,n,replace=False)])
        Z=TSNE(n_components=2,random_state=SEED,init='pca',perplexity=30).fit_transform(Xs[sel])
        scatter2d(Z, y[sel], f't-SNE 9-SFM: {ds}', os.path.join(OUTDIR,f'sep_tsne_{ds.lower()}.png'))
print('=== SEL 4 (t-SNE) SELESAI ===')

## 5. Histogram overlap 1D (proyeksi LDA) - bukti tumpang-tindih

In [ ]:
if ok:
    fig,axes=plt.subplots(1,len(data),figsize=(4.2*len(data),3.4))
    if len(data)==1: axes=[axes]
    for ax,(ds,(X,y)) in zip(axes,data.items()):
        Xs=StandardScaler().fit_transform(X)
        z=LinearDiscriminantAnalysis(n_components=1).fit_transform(Xs,y).ravel()
        ax.hist(z[y==0],bins=80,alpha=0.5,label='normal',color='#2c7fb8',density=True)
        ax.hist(z[y==1],bins=80,alpha=0.5,label='attack',color='#d7301f',density=True)
        ax.set_title(f'LDA-1D: {ds}',fontsize=10); ax.legend(fontsize=8); ax.set_xlabel('LDA score',fontsize=9)
    plt.tight_layout(); p=os.path.join(OUTDIR,'sep_overlap_hist.png'); plt.savefig(p,dpi=300); plt.close()
    print('saved',p)
print('=== SEL 5 (histogram overlap) SELESAI ===')

## 6. Unggah ke S3 (results/separability)

In [ ]:
S3_BUCKET=os.environ.get('FED_S3_BUCKET','ssh-detection-features-232032302717')
REGION=os.environ.get('AWS_REGION','ap-southeast-1'); PREFIX='unsw-far/federated/results/separability'
try:
    import boto3; s3=boto3.client('s3',region_name=REGION); up=0
    for f in os.listdir(OUTDIR):
        if f.startswith('sep_') or f=='separability_metrics.csv':
            s3.upload_file(os.path.join(OUTDIR,f), S3_BUCKET, f'{PREFIX}/{f}'); up+=1; print('  ok ->',f)
    print(f'\nunggah {up} berkas -> s3://{S3_BUCKET}/{PREFIX}/')
except Exception as e: print('(upload S3 gagal:',e,')')
print('=== SEL 6 (unggah) SELESAI ===')

## 7. Catatan untuk naskah (poin reviewer 1)

- `sep_pca_*.png` + `sep_tsne_*.png` -> Gambar baru di Bagian 5.5: tunjukkan
  normal (biru) & attack (merah) saling tumpang-tindih di ruang 9-SFM.
- `separability_metrics.csv`: AUC per-fitur (banyak ~0.5 = lemah), silhouette
  rendah, LDA-overlap tinggi -> ANGKA pendukung klaim 'kurang separable'.
- `sep_overlap_hist.png`: bukti 1D paling lugas tumpang-tindih normal vs attack.
- Semua angka/plot NYATA dari data; masukkan nilai konkret ke teks setelah run.